# BERTopic inspection (publication month 2026-01)

Evidence for selected works published 2026-01-01 through 2026-01-31: embedding coverage in Qdrant, work summaries, and discovery-topic headlines/labels.

A discovery topic label (headline) is not a weak signal by itself; it only names a candidate cluster for later weak-signal assessment.


In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"
from IPython.display import display
from qdrant_client import QdrantClient
from qdrant_client.http import models as qm
from sqlalchemy import and_, func, inspect, or_, select, text

from src.common.db import create_processor_engine
from src.common.settings import Settings
from src.corpus.select import _reconstruct_abstract
from src.models.labels import TopicLabel, WorkSummary
from src.models.topics import DiscoveryTopic, TopicRun
from src.models.upstream import work_current, work_versions

TOK_RATIO = 3.0
CONTEXT_LEN = 4096

settings = Settings()
engine = create_processor_engine(
    settings.database_url, upstream_schema=settings.upstream_schema
)
qdrant = QdrantClient(url=settings.qdrant_url)
config = settings.corpus.model_copy(
    update={
        "published_from": date(2026, 1, 1),
        "published_to": date(2026, 1, 31),
    }
)

pd.set_option("display.max_colwidth", 160)
print(f"Published window: {config.published_from} to {config.published_to}")
print(f"Qdrant collection: {settings.qdrant_collection}")


Published window: 2026-01-01 to 2026-01-31
Qdrant collection: openalex_tabstract


## Works

Selected works use the corpus filter from settings: collection scopes, work types, publication interval, excluded OpenAlex domains, a stored abstract, and no cross-scope version conflict.


In [2]:
LOAD_CACHE_WORKS_DF = True
works_cache_path = Path("data") / "works_df.parquet"
entity_ids_cache_path = Path("data") / "works_entity_ids_2026-01.parquet"

print(
    f"Scopes: {len(config.scope_ids)}. "
    f"\nTypes: {', '.join(config.work_types)}. "
    f"\nPublished: {config.published_from.isoformat()} to {config.published_to.isoformat()}. "
    f"\nExcluded domains: {', '.join(config.excluded_domain_ids) or 'none'}."
)

payload = work_versions.c.payload
work_type = payload["type"].astext
published = payload["publication_date"].astext
domain_id = payload["primary_topic"]["domain"]["id"].astext
abstract_index = payload["abstract_inverted_index"]
abstract_kind = func.jsonb_typeof(abstract_index)
joined = work_current.join(
    work_versions, work_versions.c.id == work_current.c.version_id
)
empty_object = text("'{}'::jsonb")
empty_string_object = text("'\"{}\"'::jsonb")
null_string_object = text("'\"null\"'::jsonb")
empty_quoted = text("'\"\"'::jsonb")
selected_filter = [
    work_current.c.scope_id.in_(config.scope_ids),
    work_type.in_(config.work_types),
    published >= config.published_from.isoformat(),
    published <= config.published_to.isoformat(),
    or_(domain_id.is_(None), domain_id.not_in(config.excluded_domain_ids)),
    or_(
        and_(abstract_kind == "object", abstract_index != empty_object),
        and_(
            abstract_kind == "string",
            abstract_index != empty_string_object,
            abstract_index != null_string_object,
            abstract_index != empty_quoted,
        ),
    ),
]
if len(config.scope_ids) > 1:
    versions = func.count(func.distinct(work_current.c.version_id))
    conflicts = (
        select(work_current.c.entity_id)
        .where(work_current.c.scope_id.in_(config.scope_ids))
        .group_by(work_current.c.entity_id)
        .having(versions > 1)
    )
    selected_filter.append(work_current.c.entity_id.not_in(conflicts))

title_text = payload["title"].astext
title_length = func.length(func.trim(title_text))
selected = (
    select(
        work_type.label("type"),
        title_length.label("title_length"),
        abstract_index.label("abstract_index"),
    )
    .select_from(joined)
    .where(*selected_filter)
)
entity_id_query = (
    select(work_current.c.entity_id.label("work_id"))
    .select_from(joined)
    .where(*selected_filter)
)

if LOAD_CACHE_WORKS_DF and works_cache_path.is_file():
    works_df = pd.read_parquet(works_cache_path)
    works_by_type = (
        works_df["type"].value_counts().rename_axis("type").reset_index(name="works")
    )
    print(f"Loaded works from {works_cache_path}")
elif not config.scope_ids:
    works_df = pd.DataFrame(columns=["type", "title_length", "abstract_length"])
    works_by_type = pd.DataFrame(columns=["type", "works"])
    print("No collection scopes are configured. Set PWF_CORPUS__SCOPE_IDS.")
else:
    with engine.connect() as connection:
        started = time.perf_counter()
        connection.execute(text("SET max_parallel_workers_per_gather = 0"))
        works_df = pd.read_sql(selected, connection)
        elapsed = time.perf_counter() - started
    print(f"Query duration: {elapsed:.2f} s")
    works_df["title_length"] = works_df["title_length"].fillna(0).astype(int)
    works_df["abstract_length"] = [
        len(_reconstruct_abstract(idx) or "") for idx in works_df["abstract_index"]
    ]
    works_df["abstract_index"] = [
        value if isinstance(value, str) else json.dumps(value)
        for value in works_df["abstract_index"]
    ]
    works_by_type = (
        works_df["type"].value_counts().rename_axis("type").reset_index(name="works")
    )
    works_cache_path.parent.mkdir(parents=True, exist_ok=True)
    works_df.to_parquet(works_cache_path, index=False)
    print(f"Saved works to {works_cache_path}")

if entity_ids_cache_path.is_file():
    entity_ids_df = pd.read_parquet(entity_ids_cache_path)
    print(f"Loaded entity ids from {entity_ids_cache_path}")
elif not config.scope_ids:
    entity_ids_df = pd.DataFrame(columns=["work_id"])
else:
    with engine.connect() as connection:
        started = time.perf_counter()
        connection.execute(text("SET max_parallel_workers_per_gather = 0"))
        entity_ids_df = pd.read_sql(entity_id_query, connection)
        elapsed = time.perf_counter() - started
    print(f"Entity-id query duration: {elapsed:.2f} s")
    entity_ids_cache_path.parent.mkdir(parents=True, exist_ok=True)
    entity_ids_df.to_parquet(entity_ids_cache_path, index=False)
    print(f"Saved entity ids to {entity_ids_cache_path}")

selected_works = len(works_df)
selected_work_ids = set(entity_ids_df["work_id"].astype(str))
print(f"Selected works: {selected_works}")
print(f"Selected work ids: {len(selected_work_ids)}")
display(works_by_type)


Scopes: 1. 
Types: article, preprint, conference-paper. 
Published: 2026-01-01 to 2026-01-31. 
Excluded domains: https://openalex.org/domains/2.
Loaded works from data/works_df.parquet
Loaded entity ids from data/works_entity_ids_2026-01.parquet
Selected works: 38534
Selected work ids: 38534


,type,works
0,article,33590
1,conference-paper,2880
2,preprint,2064


In [ ]:
works_df.head(3)

,type,title_length,abstract_index,abstract_length
0,conference-paper,154,"{""The"": [0, 74, 102], ""modern"": [1, 67], ""agricultural"": [2, 147], ""sector"": [3, 148], ""is"": [4, 107], ""under"": [5], ""intense"": [6], ""pressure"": [7], ""towar...",1195
1,article,145,"{""Background"": [0], ""Coexisting"": [1], ""forms"": [2, 29, 289], ""of"": [3, 30, 70, 90, 113, 143, 149, 155, 163, 244, 252, 264, 272, 290, 301, 310, 360, 367], ""...",2786
2,article,229,"{""Background"": [0], ""and"": [1, 30, 43, 78, 95, 111, 145, 191, 198, 221, 230, 237], ""Purpose"": [2], ""Glioblastoma,"": [3], ""the"": [4, 46, 170, 180], ""most"": [...",1796


## Embeddings

How many selected works for this month have a vector in the configured Qdrant collection, plus a small sample of point ids / work ids.


In [4]:
collection = settings.qdrant_collection
print(f"Collection: {collection}")
print(
    "Queried: Qdrant collection "
    f"`{collection}` with payload publication_date in "
    f"[{config.published_from}, {config.published_to}]"
)

if not qdrant.collection_exists(collection):
    collection_total = 0
    embedded_work_ids = set()
    point_samples = []
    print("Collection is not present.")
else:
    collection_total = qdrant.count(collection, exact=True).count
    month_filter = qm.Filter(
        must=[
            qm.FieldCondition(
                key="publication_date",
                range=qm.DatetimeRange(
                    gte=config.published_from.isoformat(),
                    lte=config.published_to.isoformat(),
                ),
            )
        ]
    )
    month_points = qdrant.count(collection, count_filter=month_filter, exact=True).count
    print(f"Collection point total: {collection_total}")
    print(f"Points with publication_date in month: {month_points}")

    embedded_work_ids = set()
    point_samples = []
    offset = None
    started = time.perf_counter()
    while True:
        points, offset = qdrant.scroll(
            collection_name=collection,
            scroll_filter=month_filter,
            limit=256,
            offset=offset,
            with_payload=["work_id"],
            with_vectors=False,
        )
        for point in points:
            work_id = (point.payload or {}).get("work_id")
            if work_id:
                embedded_work_ids.add(str(work_id))
            if len(point_samples) < 8:
                point_samples.append({"point_id": point.id, "work_id": work_id})
        if offset is None:
            break
    print(f"Scroll duration: {time.perf_counter() - started:.2f} s")

selected_with_vectors = len(selected_work_ids & embedded_work_ids)
print(f"Selected works with vectors: {selected_with_vectors}")
print(f"Selected works missing vectors: {len(selected_work_ids - embedded_work_ids)}")
sample_frame = pd.DataFrame(point_samples)
print("Sample point ids / work ids:")
display(sample_frame)


Collection: openalex_tabstract
Queried: Qdrant collection `openalex_tabstract` with payload publication_date in [2026-01-01, 2026-01-31]


Collection point total: 38030
Points with publication_date in month: 38030
Scroll duration: 43.92 s
Selected works with vectors: 38030
Selected works missing vectors: 504
Sample point ids / work ids:


,point_id,work_id
0,00005099-6235-5b34-bcdb-27da35f18be7,https://openalex.org/W7125911450
1,0002281d-1e98-5525-8c95-af1998991b8e,https://openalex.org/W7128804491
2,0003d573-0cb4-554f-8bb0-fefc99afde05,https://openalex.org/W7125812079
3,00050616-5438-5c85-8bf2-1cad21805505,https://openalex.org/W7122701870
4,00051202-5bdd-54ba-9ea1-7c89101caf73,https://openalex.org/W7118159315
5,000735a1-cebb-5925-ba03-ed71c5756dfe,https://openalex.org/W7125813281
6,000baa8d-de5f-58c3-ba60-91eff9ec1487,https://openalex.org/W7124286321
7,000c1a9a-95e0-58c5-9349-9d679c4c5162,https://openalex.org/W7151264759


## Topic run for this corpus

Latest row in `pwf_topic_runs` whose publication interval covers 2026-01 (prefer exact match on the notebook window).


In [5]:
inspector = inspect(engine)
topic_run = None
topic_run_id = None

if not inspector.has_table(TopicRun.__tablename__):
    print(f"Table queried: {TopicRun.__tablename__} (missing)")
    print("Topic runs covering this corpus: 0")
else:
    print(f"Table queried: {TopicRun.__tablename__}")
    covering = (
        select(TopicRun)
        .where(
            TopicRun.published_from <= config.published_from,
            TopicRun.published_to >= config.published_to,
        )
        .order_by(TopicRun.started_at.desc())
    )
    exact = (
        select(TopicRun)
        .where(
            TopicRun.published_from == config.published_from,
            TopicRun.published_to == config.published_to,
        )
        .order_by(TopicRun.started_at.desc())
    )
    with engine.connect() as connection:
        exact_rows = connection.execute(exact).mappings().all()
        covering_rows = connection.execute(covering).mappings().all()
        run_total = connection.execute(
            select(func.count()).select_from(TopicRun)
        ).scalar_one()
    print(f"All topic runs in database: {run_total}")
    print(
        f"Runs with exact window {config.published_from}..{config.published_to}: "
        f"{len(exact_rows)}"
    )
    print(f"Runs covering the window: {len(covering_rows)}")
    chosen = exact_rows[0] if exact_rows else (covering_rows[0] if covering_rows else None)
    if chosen is None:
        print("No topic run covers this corpus. Downstream counts use run_id=None.")
        display(
            pd.DataFrame(
                columns=[
                    "run_id",
                    "status",
                    "published_from",
                    "published_to",
                    "work_count",
                    "topic_count",
                    "started_at",
                ]
            )
        )
    else:
        topic_run = dict(chosen)
        topic_run_id = topic_run["run_id"]
        print(f"Latest covering topic_run_id: {topic_run_id}")
        display(
            pd.DataFrame(
                [
                    {
                        "run_id": topic_run["run_id"],
                        "status": topic_run["status"],
                        "published_from": topic_run["published_from"],
                        "published_to": topic_run["published_to"],
                        "work_count": topic_run["work_count"],
                        "topic_count": topic_run["topic_count"],
                        "outlier_count": topic_run.get("outlier_count"),
                        "started_at": topic_run["started_at"],
                        "finished_at": topic_run.get("finished_at"),
                    }
                ]
            )
        )


Table queried: pwf_topic_runs
All topic runs in database: 1
Runs with exact window 2026-01-01..2026-01-31: 1
Runs covering the window: 1
Latest covering topic_run_id: fa3ddda5-8294-48fa-a581-1e8f45ba1ac7


,run_id,status,published_from,published_to,work_count,topic_count,outlier_count,started_at,finished_at
0,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,succeeded,2026-01-01,2026-01-31,38534,280,16427,2026-09-29 16:47:53.404489+00:00,2026-09-29 16:49:36.484277+00:00


## Summaries

Rows in `pwf_work_summaries` for the latest topic run that covers this corpus, with a sample of summary text.


In [6]:
print(f"Table queried: {WorkSummary.__tablename__}")
if not inspector.has_table(WorkSummary.__tablename__):
    summary_count_for_run = 0
    print("Table missing. Summaries: 0")
    display(
        pd.DataFrame(
            columns=["topic_run_id", "discovery_topic_id", "work_id", "summary_text"]
        )
    )
elif topic_run_id is None:
    summary_count_for_run = 0
    with engine.connect() as connection:
        all_summaries = connection.execute(
            select(func.count()).select_from(WorkSummary)
        ).scalar_one()
    print(f"No covering topic run. All-rows count in table: {all_summaries}")
    print("Summaries for covering run: 0")
    display(
        pd.DataFrame(
            columns=["topic_run_id", "discovery_topic_id", "work_id", "summary_text"]
        )
    )
else:
    summary_count_stmt = (
        select(func.count())
        .select_from(WorkSummary)
        .where(WorkSummary.topic_run_id == topic_run_id)
    )
    sample_stmt = (
        select(
            WorkSummary.topic_run_id,
            WorkSummary.discovery_topic_id,
            WorkSummary.work_id,
            WorkSummary.summary_text,
        )
        .where(WorkSummary.topic_run_id == topic_run_id)
        .limit(5)
    )
    with engine.connect() as connection:
        summary_count_for_run = connection.execute(summary_count_stmt).scalar_one()
        summary_sample = pd.DataFrame(connection.execute(sample_stmt).mappings().all())
    print(f"Summaries for topic_run_id={topic_run_id}: {summary_count_for_run}")
    display(summary_sample)


Table queried: pwf_work_summaries
Summaries for topic_run_id=fa3ddda5-8294-48fa-a581-1e8f45ba1ac7: 81


,topic_run_id,discovery_topic_id,work_id,summary_text
0,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,06d5b49c-6944-42b9-b6b4-cc26d958ad34,https://openalex.org/W7155951210,This study presents a manganese-mediated radical cascade strategy for synthesizing 8-membered tetrahydroazocines. The reaction couples N-substituted enamide...
1,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,0278c702-3484-4a9e-bf66-9220652b50cb,https://openalex.org/W7213460487,"This study integrates morphological and molecular analyses to refine the taxonomy of the trematode family Didymozoidae, which primarily parasitizes marine f..."
2,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,0278c702-3484-4a9e-bf66-9220652b50cb,https://openalex.org/W7118549149,"This study investigates how the pathogenic ciliate DaScPc, responsible for mass mortality in the Caribbean sea urchin *Diadema antillarum*, persists in mari..."
3,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,06d5b49c-6944-42b9-b6b4-cc26d958ad34,https://openalex.org/W7118012694,"This study introduces a mild, metal-free intramolecular cyclization method for synthesizing diverse 4- to 8-membered N–O heterocycles. Optimized through sys..."
4,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,0278c702-3484-4a9e-bf66-9220652b50cb,https://openalex.org/W7126208274,"This study integrates molecular phylogenetics and microscopy to characterize Japanese rhizostome jellyfish, resolving key taxonomic uncertainties and clarif..."


## Discovery topics, headlines, and labels

Discovery topics in `pwf_discovery_topics` and headlines in `pwf_topic_labels` for the covering topic run, with topic size. Count how many topics have a non-null headline so a reader can judge whether labeled candidates exist after BERTopic and labeling.


In [7]:
print(f"Tables queried: {DiscoveryTopic.__tablename__}, {TopicLabel.__tablename__}")

discovery_topic_count = 0
topics_with_headline = 0
headline_rows = 0

if not inspector.has_table(DiscoveryTopic.__tablename__):
    print(f"{DiscoveryTopic.__tablename__} missing. Discovery topics: 0")
    display(
        pd.DataFrame(
            columns=[
                "discovery_topic_id",
                "topic_run_id",
                "size",
                "headline",
                "status",
            ]
        )
    )
elif topic_run_id is None:
    with engine.connect() as connection:
        discovery_topic_count = connection.execute(
            select(func.count()).select_from(DiscoveryTopic)
        ).scalar_one()
        if inspector.has_table(TopicLabel.__tablename__):
            headline_rows = connection.execute(
                select(func.count())
                .select_from(TopicLabel)
                .where(TopicLabel.headline.is_not(None))
            ).scalar_one()
    print("No covering topic run.")
    print(f"All discovery topics in database: {discovery_topic_count}")
    print(f"All non-null headlines in database: {headline_rows}")
    print("Discovery topics for covering run: 0")
    print("Topics with a headline for covering run: 0")
    display(
        pd.DataFrame(
            columns=[
                "discovery_topic_id",
                "topic_run_id",
                "size",
                "headline",
                "status",
            ]
        )
    )
else:
    topics_stmt = (
        select(
            DiscoveryTopic.discovery_topic_id,
            DiscoveryTopic.topic_run_id,
            DiscoveryTopic.size,
            TopicLabel.headline,
            TopicLabel.status,
        )
        .select_from(DiscoveryTopic)
        .outerjoin(
            TopicLabel,
            and_(
                TopicLabel.discovery_topic_id == DiscoveryTopic.discovery_topic_id,
                TopicLabel.topic_run_id == DiscoveryTopic.topic_run_id,
            ),
        )
        .where(DiscoveryTopic.topic_run_id == topic_run_id)
        .order_by(DiscoveryTopic.size.desc(), DiscoveryTopic.discovery_topic_id)
    )
    with engine.connect() as connection:
        labels_frame = pd.DataFrame(connection.execute(topics_stmt).mappings().all())
    discovery_topic_count = len(labels_frame)
    topics_with_headline = (
        int(labels_frame["headline"].notna().sum()) if len(labels_frame) else 0
    )
    headline_rows = topics_with_headline
    print(f"Discovery topics for topic_run_id={topic_run_id}: {discovery_topic_count}")
    print(f"Topics with a headline: {topics_with_headline}")
    print("Sample (largest topics first):")
    display(labels_frame.head(10))

print("---")
print(f"EVIDENCE selected_works={selected_works}")
print(f"EVIDENCE selected_with_vectors={selected_with_vectors}")
print(f"EVIDENCE summaries={summary_count_for_run}")
print(f"EVIDENCE headlines={headline_rows}")
print(f"EVIDENCE discovery_topics={discovery_topic_count}")
print(f"EVIDENCE topics_with_headline={topics_with_headline}")


Tables queried: pwf_discovery_topics, pwf_topic_labels
Discovery topics for topic_run_id=fa3ddda5-8294-48fa-a581-1e8f45ba1ac7: 280
Topics with a headline: 28
Sample (largest topics first):


,discovery_topic_id,topic_run_id,size,headline,status
0,2f12a0f2-badc-4e2f-9462-1b9a2df6dabc,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,588,NaN,NaN
1,5264b5d8-009a-474e-ad98-ea5981a46d78,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,423,NaN,NaN
2,04809f3c-f023-473e-911e-563598605900,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,416,Advancing Safety and Efficacy in Complex Cardiac Interventions,succeeded
3,421d7235-fa58-43aa-846c-2f986f56db96,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,413,NaN,NaN
4,5e5fef58-1e55-4760-9c23-3dbd64816088,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,402,NaN,NaN
5,fcbbcd15-f9d3-49ef-836c-8ac116fd32af,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,364,NaN,NaN
6,9291fca3-ff47-4183-a1a7-f29cda7b05ec,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,323,NaN,NaN
7,a27930c6-61f9-432b-8ef3-f67f80213bf4,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,316,NaN,NaN
8,fc2b885e-1181-4d7b-b43d-b1c3e9f1ccf3,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,304,NaN,NaN
9,b96e02d8-af96-460d-bb41-f8f97733b357,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,299,NaN,NaN


---
EVIDENCE selected_works=38534
EVIDENCE selected_with_vectors=38030
EVIDENCE summaries=81
EVIDENCE headlines=28
EVIDENCE discovery_topics=280
EVIDENCE topics_with_headline=28
